A full pipeline notebook, complete with orchestration and visualizations.

In [0]:
%run "/Users/nickbartram25@gmail.com/youtube_stats_databricks/youtube_etl_functions"

In [0]:
# Orchestration layer for YouTube CSV files
folder_path = "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/"

df_bronze = load_all_regions(spark, folder_path)
df_bronze = validate_bronze(df_bronze, expected_regions=ALLOWED_REGIONS)
df_silver = transform_silver(df_bronze)
df_silver = deduplicate_trending(df_silver)
df_silver = validate_silver(df_silver, df_bronze)

In [0]:
# Orchestration: creating necesary variables for JSON files
folder_path = "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/"

category_raw_dfs   = load_all_category_jsons(spark, folder_path)
category_valid_dfs = validate_category_raw(category_raw_dfs)

In [0]:
# Build category dimension df
category_dim_df = build_category_dim(category_valid_dfs)

# Join silver and category
df_gold = df_silver.join(category_dim_df, "category_id", "left")

In [0]:
# Add business metrics
df_gold = add_business_metrics(df_gold)

# Finaliza gold schema
df_gold = finalize_gold_schema(df_gold)

# Validate gold DataFrame
df_gold = validate_gold(df_gold, df_silver)

# Display df_gold
df_gold.display()

In [0]:
# Write table to delta
write_gold_table(df_gold, "youtube_gold")